<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/20-efficient-inference-deployment.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Efficient Inference and Deployment** {#efficient-inference-deployment}

Inference is the process that turns a trained model into decisions under an operational contract. A notebook asks whether the model can produce the right tensor; a deployed system must also validate requests, meet latency objectives, share finite memory across users, survive malformed inputs, expose telemetry, and support rollback. Consequently, "faster inference" is not one optimization. It is a joint design problem over **model quality, request scheduling, numerical representation, runtime compilation, hardware, and operations**.

This chapter continues the UCI Optical Recognition of Handwritten Digits workload from Chapter 19. The [UCI dataset](https://doi.org/10.24432/C50P49) contains 1,797 labeled 8 by 8 images and is licensed **CC BY 4.0**. One deterministic train/validation/test split supports the classifier deployment experiments. For the autoregressive mechanisms, each image becomes a sequence consisting of a beginning token, its class token, 64 integer pixel-intensity tokens, and an end token. That derived sequence remains tied to the same observations; it is a mechanism test for caching and exact speculative sampling, not a language-model benchmark.

<details>
<summary><strong>PyTorch: establish the shared Digits deployment workload</strong></summary>

```python
import copy
import math
import random
import time

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2020):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
train_ids, remaining_ids = train_test_split(
    all_ids, test_size=0.30, stratify=digits.target, random_state=2020
)
val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    stratify=digits.target[remaining_ids],
    random_state=2020,
)

# UCI documents pixel intensities on 0..16, so this fixed divisor does not leak test statistics.
features = torch.tensor(digits.data / 16.0, dtype=torch.float32)
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(features[train_ids], targets[train_ids])
val_dataset = TensorDataset(features[val_ids], targets[val_ids])
test_dataset = TensorDataset(features[test_ids], targets[test_ids])
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(2020),
)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class DigitMLP(nn.Module):
    def __init__(self, hidden1=128, hidden2=64):
        super().__init__()
        self.fc1 = nn.Linear(64, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.inference_mode():
        for x, y in loader:
            correct += int((model(x).argmax(1) == y).sum())
            total += len(y)
    return correct / total


seed_everything()
baseline_model = DigitMLP()
optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(15):
    baseline_model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(baseline_model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Reuse the same observations as short discrete sequences for generative mechanisms.
BOS, LABEL_OFFSET, EOS, VOCAB_SIZE = 27, 17, 28, 29
pixel_tokens = torch.tensor(digits.data.astype(np.int64), dtype=torch.long)
token_sequences = torch.cat(
    [
        torch.full((len(digits.data), 1), BOS, dtype=torch.long),
        targets[:, None] + LABEL_OFFSET,
        pixel_tokens,
        torch.full((len(digits.data), 1), EOS, dtype=torch.long),
    ],
    dim=1,
)

baseline_accuracy = accuracy(baseline_model, test_loader)
assert len(set(train_ids) & set(test_ids)) == 0
assert token_sequences.shape == (1797, 67) and baseline_accuracy > 0.92
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "test_accuracy": round(baseline_accuracy, 3)})
```

</details>

The baseline is intentionally small enough to execute on CPU. All timings below describe this local eager environment unless a production runtime is explicitly named; they demonstrate measurement and decision procedures rather than universal hardware rankings.


### **The Inference Lifecycle** {#inference-lifecycle}

A forward pass maps a valid tensor to logits. An inference lifecycle begins earlier and ends later: a request is admitted, authenticated and parsed; preprocessing reproduces the training contract; a scheduler forms executable work; the runtime selects kernels and device memory; postprocessing converts tensors into an application response; telemetry records what happened. A correct model behind an inconsistent tokenizer, stale normalization rule, or unbounded request shape is an incorrect service.

![The inference lifecycle from request validation through observation.](assets/dl20-inference-lifecycle.svg){fig-align="center" width="76%" fig-alt="A five-stage flow moves from request validation to preparation, model execution, postprocessing, and observation, with a release contract below."}

*Original teaching diagram based on the deployment boundaries described by [ONNX Runtime](https://onnxruntime.ai/docs/) and the request-serving responsibilities in [NVIDIA Triton Inference Server](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/).* 

A useful interface contract specifies at least the input schema, accepted shapes and dtypes, preprocessing version, output semantics, model artifact digest, timeout, and failure behavior. Idempotent request identifiers make retries observable. A readiness check should load the actual artifact and run a representative input; a liveness check should only establish that the process can respond, otherwise a transient accelerator failure may trigger a restart storm.

<details>
<summary><strong>PyTorch: wrap the model in an explicit request-response contract</strong></summary>

```python
MODEL_VERSION = "digits-mlp-2020-v1"


def predict_digits(request):
    # 1. Validate the external schema before allocating an unbounded tensor.
    request_id = str(request["request_id"])
    raw = np.asarray(request["pixels"], dtype=np.float32)
    if raw.shape not in {(8, 8), (64,)}:
        raise ValueError("pixels must have shape (8, 8) or (64,)")
    if not np.isfinite(raw).all() or raw.min() < 0 or raw.max() > 16:
        raise ValueError("pixels must be finite values in the documented 0..16 range")

    # 2. Apply the same fixed preprocessing contract used during training.
    model_input = torch.from_numpy(raw.reshape(1, 64) / 16.0)

    # 3. Disable autograd and execute the immutable evaluation model.
    started_ns = time.perf_counter_ns()
    baseline_model.eval()
    with torch.inference_mode():
        probabilities = baseline_model(model_input).softmax(dim=-1)[0]
    elapsed_ms = (time.perf_counter_ns() - started_ns) / 1e6

    # 4. Return stable application semantics and enough identity for tracing.
    top_probability, top_class = probabilities.max(dim=0)
    return {
        "request_id": request_id,
        "model_version": MODEL_VERSION,
        "prediction": int(top_class),
        "confidence": float(top_probability),
        "model_ms": elapsed_ms,
    }


sample_id = int(test_ids[0])
response = predict_digits({"request_id": "demo-1", "pixels": digits.images[sample_id]})
assert response["prediction"] in range(10) and 0.0 <= response["confidence"] <= 1.0
try:
    predict_digits({"request_id": "bad", "pixels": np.zeros((16, 16))})
    raise AssertionError("shape validation should have failed")
except ValueError:
    pass
print(response)
```

</details>

The wrapper separates **model latency** from end-to-end latency. That distinction is diagnostic: a slow response may be caused by queueing or image decoding even when the model kernel is healthy. The same contract should be replayed against every candidate artifact before traffic is shifted.


### **Latency, Throughput, and Tail Behavior** {#latency-throughput-tail}

Latency is elapsed time for one request, while throughput is completed work per unit time. For request (i),

$$L_i = t_i^{\text{return}} - t_i^{\text{arrival}}, \qquad \lambda = \frac{N_{\text{completed}}}{\Delta t}.$$

A service-level objective is normally a quantile such as (P(L \leq L_{\mathrm{SLO}}) \geq 0.99), not a mean. The 99th percentile captures a slow tail that may be invisible in the average. Queueing couples the metrics: when offered load approaches service capacity, small bursts can create disproportionate waiting time. Little's law, (Q=\lambda W), relates average in-system requests (Q), achieved throughput (lambda), and average time (W) for a stable system.

![A cumulative latency curve marking median and tail latency.](assets/dl20-latency-tail.svg){fig-align="center" width="72%" fig-alt="A cumulative latency curve marks p50 and p99, with queueing, shape fallback, and cache misses identified as tail causes."}

For autoregressive generation, one scalar is insufficient. **Time to first token (TTFT)** covers queueing plus prompt prefill, **inter-token latency (ITL)** or **time per output token (TPOT)** describes decode cadence, and end-to-end latency also depends on output length. Interactive systems may prioritize TTFT and tail ITL; offline jobs often prioritize tokens per second.

<details>
<summary><strong>Python: measure warm, synchronized latency distributions and throughput</strong></summary>

```python
def benchmark_classifier(model, requests, batch_size=1, repeats=3):
    model.eval()
    # Warm-up removes one-time initialization from the steady-state sample.
    with torch.inference_mode():
        for _ in range(20):
            model(requests[: min(batch_size, len(requests))])

    samples_ms = []
    started = time.perf_counter()
    completed = 0
    with torch.inference_mode():
        for _ in range(repeats):
            for start in range(0, len(requests), batch_size):
                batch = requests[start : start + batch_size]
                tick = time.perf_counter_ns()
                model(batch)
                samples_ms.append((time.perf_counter_ns() - tick) / 1e6)
                completed += len(batch)
    elapsed = time.perf_counter() - started
    return {
        "p50_batch_ms": float(np.percentile(samples_ms, 50)),
        "p95_batch_ms": float(np.percentile(samples_ms, 95)),
        "p99_batch_ms": float(np.percentile(samples_ms, 99)),
        "requests_per_second": completed / elapsed,
    }


test_x = features[test_ids]
single_metrics = benchmark_classifier(baseline_model, test_x[:128], batch_size=1)
batch_metrics = benchmark_classifier(baseline_model, test_x[:128], batch_size=32)
assert single_metrics["p99_batch_ms"] >= single_metrics["p50_batch_ms"]
assert batch_metrics["requests_per_second"] > 0
print({"single": single_metrics, "batch_32": batch_metrics})
```

</details>

Timings must record hardware, runtime version, thread count, batch/sequence shape, precision, warm-up, concurrency, and whether device execution was synchronized. Compare distributions under the expected arrival process; a tight microbenchmark can expose kernel cost but cannot predict network queueing or multi-tenant interference.


### **Static, Dynamic, and Continuous Batching** {#static-dynamic-continuous-batching}

Batching amortizes dispatch overhead and exposes parallel work, but it can delay an early request while the scheduler waits for companions. **Static batching** fixes the batch shape before execution and is simple for homogeneous offline data. **Dynamic batching** collects compatible requests until a maximum size or queue timeout is reached. **Continuous batching** is designed for variable-length autoregressive decoding: completed sequences leave slots and waiting sequences enter without waiting for the entire original batch to finish.

![Timelines comparing static, dynamic, and continuous batching.](assets/dl20-batching.svg){fig-align="center" width="76%" fig-alt="Three timelines show fixed full batches, batches assembled by size or timeout, and continuously refilled sequence slots."}

*Original teaching diagram based on the [Triton dynamic batcher](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html) and iteration-level scheduling used in modern generative serving systems.*

Compatibility matters as much as arrival time. Requests may require the same model version, precision, adapter, tensor shape bucket, or decoding policy. Padding a highly variable batch wastes compute; over-fragmenting queues sacrifices utilization. The scheduler therefore optimizes a constrained objective such as throughput subject to (p99 \leq L_{\mathrm{SLO}}), not batch size alone.

<details>
<summary><strong>Python: simulate dynamic and continuous schedulers with Digits-derived requests</strong></summary>

```python
# Deterministic inter-arrival gaps and generation lengths come from test observations.
arrival_ms = np.cumsum(0.15 + (digits.target[test_ids[:80]] % 4) * 0.08)
token_budgets = 4 + (pixel_tokens[test_ids[:80]] > 0).sum(dim=1).numpy() // 4


def dynamic_batches(arrivals, max_batch=12, timeout_ms=0.8):
    batches, queue = [], []
    first_arrival = None
    for request_id, arrival in enumerate(arrivals):
        if queue and arrival - first_arrival >= timeout_ms:
            batches.append(queue)
            queue, first_arrival = [], None
        if not queue:
            first_arrival = arrival
        queue.append(request_id)
        if len(queue) == max_batch:
            batches.append(queue)
            queue, first_arrival = [], None
    if queue:
        batches.append(queue)
    return batches


def continuous_decode(lengths, slots=8):
    waiting = list(enumerate(lengths))
    active, finish_step = [], {}
    step = 0
    while waiting or active:
        while waiting and len(active) < slots:
            request_id, length = waiting.pop(0)
            active.append([request_id, int(length)])
        step += 1
        for item in active:
            item[1] -= 1
        finished = [item for item in active if item[1] == 0]
        for request_id, _ in finished:
            finish_step[request_id] = step
        active = [item for item in active if item[1] > 0]
    return finish_step, step


formed = dynamic_batches(arrival_ms)
continuous_finish, continuous_steps = continuous_decode(token_budgets, slots=8)
padded_steps = sum(max(token_budgets[i : i + 8]) for i in range(0, len(token_budgets), 8))
assert sum(map(len, formed)) == len(arrival_ms)
assert set(continuous_finish) == set(range(len(token_budgets)))
print({
    "dynamic_batch_sizes": [len(batch) for batch in formed[:6]],
    "continuous_steps": continuous_steps,
    "fixed_padded_steps": int(padded_steps),
})
```

</details>

Continuous batching does not make every request faster. It improves slot utilization, while admission control and preemption decide who waits. Report queue delay separately from execution time, and inspect latency by prompt length, output length, adapter, and priority class so a favorable aggregate does not hide starvation.


### **KV Cache and Memory Management** {#kv-cache-memory-management}

In causal self-attention, a token at decode step (t) needs keys and values from positions (1,\ldots,t). Recomputing every old projection after each new token repeats work. A **KV cache** stores previous key/value tensors after prefill and appends one position per decode step. The new query still attends over the prefix, but old keys and values are reused.

For (N_L) layers, batch size (B), cached length (T), (H_{kv}) key/value heads, head dimension (d_h), and (b) bytes per element, the dominant cache size is

$$M_{KV} = 2N_LBTH_{kv}d_hb.$$

The factor 2 represents keys and values. Multi-query attention (one KV head) and grouped-query attention (fewer KV heads than query heads) reduce this memory term. Static caches reserve a known maximum shape and are compilation friendly but may waste capacity; dynamic caches grow naturally but complicate allocation. [Hugging Face's cache guide](https://huggingface.co/docs/transformers/main/kv_cache) documents this practical speed-memory trade-off.

![KV cache reuse and paged memory management.](assets/dl20-kv-cache.svg){fig-align="center" width="76%" fig-alt="A prefill stage writes key and value blocks to a paged cache, and each decode step reads the prefix and appends one block."}

Variable sequence lengths cause fragmentation when each request receives a large contiguous reservation. [PagedAttention](https://arxiv.org/abs/2309.06180) maps logical KV blocks to non-contiguous physical blocks, analogous to virtual-memory paging. It improves allocation and prefix sharing; it does not remove the linear growth of cache content with context length.

<details>
<summary><strong>PyTorch: verify full causal attention against incremental KV caching</strong></summary>

```python
seed_everything(2020)
sequence = token_sequences[int(test_ids[0]), :18]
model_dim, heads, head_dim = 32, 4, 8
embedding = nn.Embedding(VOCAB_SIZE, model_dim)
q_proj = nn.Linear(model_dim, model_dim, bias=False)
k_proj = nn.Linear(model_dim, model_dim, bias=False)
v_proj = nn.Linear(model_dim, model_dim, bias=False)


def split_heads(x):
    batch, length, _ = x.shape
    return x.view(batch, length, heads, head_dim).transpose(1, 2)


hidden = embedding(sequence[None, :])
q_full, k_full, v_full = map(split_heads, (q_proj(hidden), k_proj(hidden), v_proj(hidden)))
scores = q_full @ k_full.transpose(-1, -2) / math.sqrt(head_dim)
causal_mask = torch.triu(torch.ones(len(sequence), len(sequence), dtype=torch.bool), diagonal=1)
full_output = torch.softmax(scores.masked_fill(causal_mask, -torch.inf), dim=-1) @ v_full

cached_k = cached_v = None
incremental_outputs = []
for position in range(len(sequence)):
    token_hidden = embedding(sequence[position : position + 1][None, :])
    q_t = split_heads(q_proj(token_hidden))
    k_t = split_heads(k_proj(token_hidden))
    v_t = split_heads(v_proj(token_hidden))
    cached_k = k_t if cached_k is None else torch.cat([cached_k, k_t], dim=2)
    cached_v = v_t if cached_v is None else torch.cat([cached_v, v_t], dim=2)
    attention_t = torch.softmax(q_t @ cached_k.transpose(-1, -2) / math.sqrt(head_dim), dim=-1)
    incremental_outputs.append(attention_t @ cached_v)

incremental_output = torch.cat(incremental_outputs, dim=2)


def kv_cache_mib(layers, batch, tokens, kv_heads, dimension, bytes_per_value):
    return 2 * layers * batch * tokens * kv_heads * dimension * bytes_per_value / 2**20


assert torch.allclose(full_output, incremental_output, atol=1e-6)
print({
    "equivalent": True,
    "example_cache_MiB": round(kv_cache_mib(32, 8, 4096, 8, 128, 2), 1),
})
```

</details>

Cache correctness must include position indices, attention masks, beam reordering, prefix ownership, and eviction. Operational diagnostics include allocated versus used blocks, cache-hit rate, fragmentation, eviction rate, and bytes per active token. A cache policy that raises batch capacity can still hurt latency if offloading or eviction causes repeated transfers.


### **Speculative Decoding** {#speculative-decoding}

Autoregressive decoding is often memory-bandwidth bound because a large target model is invoked for one new token at a time. **Speculative decoding** asks a cheaper draft model (q) to propose several tokens, then uses the target model (p) to verify the proposal in parallel. The essential requirement is distributional exactness: rejected draft tokens must be corrected so outputs still follow (p), not an approximation chosen only for speed.

For one proposed token (y\sim q), accept it with

$$\alpha(y)=\min\left(1,\frac{p(y)}{q(y)}\right).$$

If it is rejected, sample from the normalized residual (r(y)\propto\max(0,p(y)-q(y))). This acceptance-rejection identity makes the marginal output exactly (p). The [original speculative decoding paper](https://proceedings.mlr.press/v202/leviathan23a.html) extends the idea to blocks and verifies them with fewer sequential target calls.

![Draft proposal, target verification, and exact correction in speculative decoding.](assets/dl20-speculative.svg){fig-align="center" width="74%" fig-alt="A draft model proposes a token block, the target model verifies it using acceptance ratios, and accepted tokens or a corrected sample extend the committed prefix."}

<details>
<summary><strong>Python: verify exact one-step speculative sampling on Digits token transitions</strong></summary>

```python
# Build a target bigram distribution and a cheaper smoothed draft from training sequences only.
transition_counts = np.full((VOCAB_SIZE, VOCAB_SIZE), 0.5, dtype=np.float64)
unigram_counts = np.full(VOCAB_SIZE, 0.5, dtype=np.float64)
for sequence_row in token_sequences[train_ids].numpy():
    np.add.at(transition_counts, (sequence_row[:-1], sequence_row[1:]), 1)
    np.add.at(unigram_counts, sequence_row[1:], 1)
target_bigram = transition_counts / transition_counts.sum(axis=1, keepdims=True)
unigram = unigram_counts / unigram_counts.sum()
draft_bigram = 0.82 * target_bigram + 0.18 * unigram[None, :]


def speculative_one_step(p, q, rng):
    proposal = int(rng.choice(len(q), p=q))
    acceptance = min(1.0, p[proposal] / q[proposal])
    if rng.random() <= acceptance:
        return proposal, True
    residual = np.maximum(p - q, 0.0)
    residual /= residual.sum()
    return int(rng.choice(len(p), p=residual)), False


context = int(token_sequences[int(test_ids[0]), 8])
p, q = target_bigram[context], draft_bigram[context]
rng = np.random.default_rng(2020)
draws, accepted = zip(*(speculative_one_step(p, q, rng) for _ in range(20000)))
empirical = np.bincount(draws, minlength=VOCAB_SIZE) / len(draws)
total_variation = 0.5 * np.abs(empirical - p).sum()
assert total_variation < 0.035
print({"acceptance_rate": round(np.mean(accepted), 3), "target_TV_error": round(total_variation, 4)})
```

</details>

Speedup depends on acceptance length, draft cost, target verification efficiency, sampling settings, and synchronization overhead. A draft that is too weak rejects frequently; one that is too large saves too little work. Measure accepted tokens per target call and end-to-end TTFT/TPOT under the actual decoding policy. Greedy assisted decoding is simpler, but it should not be confused with the exact sampling algorithm above.


### **Post-Training Quantization** {#post-training-quantization}

Quantization represents real values with a smaller discrete code. An affine mapping uses scale (s>0), zero point (z), and integer range ([q_{\min},q_{\max}]):

$$q=\operatorname{clip}\left(\operatorname{round}(x/s)+z,q_{\min},q_{\max}\right), \qquad \hat{x}=s(q-z).$$

(q) is stored or processed by an integer kernel and (hat{x}) is the reconstructed value. Symmetric signed quantization sets (z=0) and usually chooses (s=\max|x|/(2^{b-1}-1)) for (b) bits. Per-channel weight scales preserve rows with different magnitudes; per-tensor activation scales are cheaper to manage. Clipping reduces range but can improve precision for the bulk of a distribution.

**Post-training quantization (PTQ)** changes a trained model without optimizing its task loss again. Dynamic PTQ estimates activation parameters at execution time. Static PTQ runs representative **training/calibration** data in advance and embeds activation parameters in the artifact. [ONNX Runtime's quantization guide](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html) distinguishes these modes and recommends comparing matched FP32 and quantized activations when quality regresses.

![Post-training calibration, conversion, and validation.](assets/dl20-ptq.svg){fig-align="center" width="76%" fig-alt="A flow moves from a floating-point model through calibration and quantization to validation, with an activation-level debug path."}

<details>
<summary><strong>PyTorch: implement calibrated W8A8 post-training quantization</strong></summary>

```python
def symmetric_scale(x, bits=8, dim=None):
    limit = 2 ** (bits - 1) - 1
    maximum = x.detach().abs().amax(dim=dim, keepdim=dim is not None)
    return maximum.clamp_min(1e-8) / limit


def quantize_dequantize(x, scale, bits=8):
    limit = 2 ** (bits - 1) - 1
    return torch.round(x / scale).clamp(-limit, limit) * scale


# Observe post-activation ranges using training data only.
activation_max = {"input": 0.0, "hidden1": 0.0, "hidden2": 0.0}
baseline_model.eval()
with torch.inference_mode():
    for x, _ in train_loader:
        h1 = F.gelu(baseline_model.fc1(x))
        h2 = F.gelu(baseline_model.fc2(h1))
        activation_max["input"] = max(activation_max["input"], float(x.abs().max()))
        activation_max["hidden1"] = max(activation_max["hidden1"], float(h1.abs().max()))
        activation_max["hidden2"] = max(activation_max["hidden2"], float(h2.abs().max()))
activation_scales = {name: torch.tensor(value / 127.0) for name, value in activation_max.items()}


class StaticPTQDigitMLP(nn.Module):
    def __init__(self, source, scales):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = [copy.deepcopy(layer) for layer in (source.fc1, source.fc2, source.fc3)]
        self.scales = scales
        with torch.no_grad():
            for layer in (self.fc1, self.fc2, self.fc3):
                row_scale = symmetric_scale(layer.weight, bits=8, dim=1)
                layer.weight.copy_(quantize_dequantize(layer.weight, row_scale, bits=8))

    def forward(self, x):
        x = quantize_dequantize(x, self.scales["input"], bits=8)
        x = F.gelu(self.fc1(x))
        x = quantize_dequantize(x, self.scales["hidden1"], bits=8)
        x = F.gelu(self.fc2(x))
        x = quantize_dequantize(x, self.scales["hidden2"], bits=8)
        return self.fc3(x)


ptq_model = StaticPTQDigitMLP(baseline_model, activation_scales)
with torch.inference_mode():
    reference_logits = baseline_model(test_x[:128])
    ptq_logits = ptq_model(test_x[:128])
ptq_accuracy = accuracy(ptq_model, test_loader)
ptq_logit_mae = float((reference_logits - ptq_logits).abs().mean())
assert ptq_accuracy > 0.90 and ptq_logit_mae > 0
print({"FP32": round(baseline_accuracy, 3), "PTQ_W8A8": round(ptq_accuracy, 3), "logit_MAE": round(ptq_logit_mae, 4)})
```

</details>

This model stores dequantized tensors so the example runs without a backend-specific integer package; it validates quantization numerics, not INT8 speed. Real acceleration requires supported packed operators and hardware instructions. Calibration data must represent deployment ranges but remain isolated from the test set; overly narrow ranges saturate shifted inputs, while outliers can waste most integer levels.


### **Quantization-Aware Training** {#quantization-aware-training}

PTQ can fail when small rounding changes accumulate through sensitive layers. **Quantization-aware training (QAT)** exposes the model to those errors before deployment. The forward pass inserts fake quantize/dequantize operations, while trainable parameters remain floating point. Because rounding has zero derivative almost everywhere, the common straight-through estimator (STE) uses an approximate identity gradient through the quantizer.

![The fake-quantized forward and straight-through backward loop in QAT.](assets/dl20-qat.svg){fig-align="center" width="74%" fig-alt="A loop shows fake-quantized weights and activations in the forward pass, task loss, straight-through gradients, and floating-point parameter updates."}

QAT does not guarantee a deployable speedup. The fake-quant graph must eventually convert to operators supported by the target backend, with the same observer, granularity, and numeric assumptions used during training. Current PyTorch quantization development is centralized in [torchao](https://docs.pytorch.org/ao/stable/workflows/qat.html); the API surface evolves, but the prepare-train-convert lifecycle remains the conceptual invariant.

<details>
<summary><strong>PyTorch: fine-tune the Digits model with straight-through fake quantization</strong></summary>

```python
def ste_quantize_dequantize(x, scale, bits=8):
    reconstructed = quantize_dequantize(x, scale, bits)
    return x + (reconstructed - x).detach()


class QATDigitMLP(DigitMLP):
    def __init__(self, source, scales):
        super().__init__()
        self.load_state_dict(copy.deepcopy(source.state_dict()))
        self.scales = scales

    def quantized_linear(self, x, layer):
        weight_scale = symmetric_scale(layer.weight, bits=8, dim=1)
        weight = ste_quantize_dequantize(layer.weight, weight_scale, bits=8)
        return F.linear(x, weight, layer.bias)

    def forward(self, x):
        x = ste_quantize_dequantize(x, self.scales["input"], bits=8)
        x = F.gelu(self.quantized_linear(x, self.fc1))
        x = ste_quantize_dequantize(x, self.scales["hidden1"], bits=8)
        x = F.gelu(self.quantized_linear(x, self.fc2))
        x = ste_quantize_dequantize(x, self.scales["hidden2"], bits=8)
        return self.quantized_linear(x, self.fc3)


qat_model = QATDigitMLP(baseline_model, activation_scales)
qat_optimizer = torch.optim.AdamW(qat_model.parameters(), lr=2e-4, weight_decay=1e-4)
for _ in range(4):
    qat_model.train()
    for x, y in train_loader:
        qat_loss = F.cross_entropy(qat_model(x), y)
        qat_optimizer.zero_grad()
        qat_loss.backward()
        qat_optimizer.step()

qat_accuracy = accuracy(qat_model, test_loader)
assert qat_accuracy > 0.90 and any(parameter.grad is not None for parameter in qat_model.parameters())
print({"PTQ": round(ptq_accuracy, 3), "QAT_fake_quant": round(qat_accuracy, 3)})
```

</details>

QAT is justified when PTQ misses a quality budget and representative fine-tuning data is available. Diagnose by layerwise activation error, clipping frequency, and per-slice task metrics before increasing training cost. If only a few operators are sensitive, mixed precision or selective quantization can be a cleaner solution than QAT everywhere.


### **Weight-Only Quantization and Low-Bit Methods** {#weight-only-low-bit}

In large models, reading weights from memory can dominate decode cost. **Weight-only quantization** stores weights at low precision while keeping activations and accumulation in a wider type. It avoids activation calibration and can reduce bandwidth, but every matrix multiplication needs a compatible packed kernel that dequantizes or directly consumes low-bit blocks.

Groupwise quantization partitions each row into blocks of (G) weights and assigns one scale per block. For (n_w) weights, (b)-bit codes, and (n_s\) FP32 scales, the logical storage is approximately

$$M \approx \frac{bn_w}{8} + 4n_s \text{ bytes}.$$

Smaller groups adapt better to local ranges but add scale metadata. Round-to-nearest (RTN) is data-free; GPTQ approximately minimizes layer reconstruction error using calibration activations; AWQ uses activation information to protect salient weight channels. Their names describe different optimization criteria, not interchangeable bit formats.

![Groupwise low-bit weight storage with one scale per block.](assets/dl20-weight-only.svg){fig-align="center" width="76%" fig-alt="A floating-point matrix is partitioned into colored groups and packed into INT4 codes plus scales, with the granularity trade-off noted."}

<details>
<summary><strong>PyTorch: implement groupwise INT4 weight reconstruction and storage accounting</strong></summary>

```python
def groupwise_weight_qdq(weight, bits=4, group_size=32):
    output_dim, input_dim = weight.shape
    padding = (-input_dim) % group_size
    padded = F.pad(weight, (0, padding))
    grouped = padded.view(output_dim, -1, group_size)
    scale = symmetric_scale(grouped, bits=bits, dim=2)
    reconstructed = quantize_dequantize(grouped, scale, bits=bits)
    return reconstructed.view(output_dim, -1)[:, :input_dim], scale


int4_model = copy.deepcopy(baseline_model)
int4_scale_count = 0
with torch.no_grad():
    for layer in (int4_model.fc1, int4_model.fc2, int4_model.fc3):
        reconstructed, scales = groupwise_weight_qdq(layer.weight, bits=4, group_size=32)
        layer.weight.copy_(reconstructed)
        int4_scale_count += scales.numel()

weight_count = sum(layer.weight.numel() for layer in (baseline_model.fc1, baseline_model.fc2, baseline_model.fc3))
bias_count = sum(layer.bias.numel() for layer in (baseline_model.fc1, baseline_model.fc2, baseline_model.fc3))
fp32_bytes = 4 * (weight_count + bias_count)
int4_logical_bytes = weight_count / 2 + 4 * (int4_scale_count + bias_count)
int4_accuracy = accuracy(int4_model, test_loader)
assert int4_logical_bytes < fp32_bytes and int4_accuracy > 0.85
print({
    "FP32_KiB": round(fp32_bytes / 1024, 1),
    "INT4_logical_KiB": round(int4_logical_bytes / 1024, 1),
    "INT4_accuracy": round(int4_accuracy, 3),
})
```

</details>

The code deliberately reconstructs FP32 weights, so its measured latency cannot represent a packed INT4 kernel. A low-bit artifact may even run slower when the runtime inserts conversions or falls back to generic operators. Validate perplexity/task metrics, long-context stability, outlier layers, memory resident set, and target-hardware tokens per second before choosing a method.


### **Pruning and Knowledge Distillation** {#pruning-knowledge-distillation}

**Pruning** sets parameters or structures to zero. Unstructured magnitude pruning can achieve high sparsity while retaining tensor shapes; without sparse kernels and favorable metadata overhead, it saves neither dense FLOPs nor wall time. Structured pruning removes channels, heads, blocks, or layers so standard dense kernels see genuinely smaller shapes, but each removal perturbs a larger functional unit.

**Knowledge distillation** trains a smaller student to match a teacher as well as the labels. With temperature (\tau), teacher logits (z_t), student logits (z_s), hard-label loss (\mathcal{L}_{CE}), and mixture weight (\alpha), a common objective is

$$\mathcal{L}=\alpha\mathcal{L}_{CE}(z_s,y)+(1-\alpha)\tau^2\,D_{KL}\!\left(\operatorname{softmax}(z_t/\tau)\;\|\;\operatorname{softmax}(z_s/\tau)\right).$$

The soft distribution communicates class similarities that a one-hot label omits. The (\tau^2) factor keeps gradient magnitudes comparable after temperature scaling.

![Pruning and distillation paths from a trained teacher.](assets/dl20-pruning-distillation.svg){fig-align="center" width="74%" fig-alt="A teacher branches into a sparse pruned model and a smaller distilled student, both of which must be measured for quality, storage, and backend latency."}

<details>
<summary><strong>PyTorch: compare unstructured pruning with a distilled dense student</strong></summary>

```python
pruned_model = copy.deepcopy(baseline_model)
prune_fraction = 0.50
all_magnitudes = torch.cat([layer.weight.detach().abs().flatten() for layer in (pruned_model.fc1, pruned_model.fc2, pruned_model.fc3)])
threshold = torch.quantile(all_magnitudes, prune_fraction)
zero_weights = total_weights = 0
with torch.no_grad():
    for layer in (pruned_model.fc1, pruned_model.fc2, pruned_model.fc3):
        mask = layer.weight.abs() > threshold
        layer.weight.mul_(mask)
        zero_weights += int((layer.weight == 0).sum())
        total_weights += layer.weight.numel()


class StudentMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 48)
        self.fc2 = nn.Linear(48, 10)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


seed_everything(2020)
student_model = StudentMLP()
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=2e-3, weight_decay=1e-4)
temperature, hard_weight = 2.0, 0.45
baseline_model.eval()
for _ in range(15):
    student_model.train()
    for x, y in train_loader:
        with torch.inference_mode():
            teacher_logits = baseline_model(x)
        student_logits = student_model(x)
        hard_loss = F.cross_entropy(student_logits, y)
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / temperature, dim=-1),
            F.softmax(teacher_logits / temperature, dim=-1),
            reduction="batchmean",
        ) * temperature**2
        distillation_loss = hard_weight * hard_loss + (1 - hard_weight) * soft_loss
        student_optimizer.zero_grad()
        distillation_loss.backward()
        student_optimizer.step()

pruned_accuracy = accuracy(pruned_model, test_loader)
student_accuracy = accuracy(student_model, test_loader)
student_parameters = sum(parameter.numel() for parameter in student_model.parameters())
teacher_parameters = sum(parameter.numel() for parameter in baseline_model.parameters())
assert zero_weights / total_weights >= 0.49 and student_parameters < teacher_parameters
print({
    "pruned_accuracy": round(pruned_accuracy, 3),
    "sparsity": round(zero_weights / total_weights, 3),
    "student_accuracy": round(student_accuracy, 3),
    "student_parameter_ratio": round(student_parameters / teacher_parameters, 3),
})
```

</details>

Pruning preserves the original architecture but asks the runtime to exploit sparsity. Distillation pays training cost to create a smaller dense graph that ordinary kernels can accelerate. Compare a student trained only with hard labels to prove that distillation, not merely architecture size, provided the benefit; then evaluate rare classes and calibration because a student may copy teacher errors.


### **Export, Compilation, and Runtime Optimization** {#export-compilation-runtime-optimization}

Deployment crosses three related but distinct boundaries. **Export** captures a program and its input constraints in a portable or ahead-of-time representation. **Compilation** transforms that graph for a target: it decomposes operators, propagates constants, fuses patterns, plans memory, and selects kernels. The **runtime** loads the artifact, binds inputs, manages devices, executes kernels, and reports failures and metrics.

![Stages from an eager model to a monitored target runtime.](assets/dl20-export-runtime.svg){fig-align="center" width="76%" fig-alt="An eager model is captured, transformed, compiled, and loaded by a runtime, while a parity gate checks representative inputs and shape constraints."}

Export makes assumptions explicit. A branch depending on arbitrary Python state may not be capturable; a dynamic dimension needs a declared range; an unsupported custom operator needs a lowering or runtime implementation. Graph fusion reduces launch and intermediate-memory overhead, but altered operation order can change floating-point rounding. Therefore every compiled artifact is a new numerical implementation that must pass parity and task-quality gates.

[PyTorch AOTInductor](https://docs.pytorch.org/docs/main/user_guide/torch_compiler/torch.compiler_aot_inductor.html) uses `torch.export` as an ahead-of-time graph boundary and can package artifacts for non-Python deployment. `torch.compile` instead optimizes execution while preserving a Python-facing workflow and may recompile when guards fail.

<details>
<summary><strong>PyTorch: capture an export graph and verify compiled parity</strong></summary>

```python
export_input = test_x[:16]
baseline_model.eval()
exported_program = torch.export.export(baseline_model, (export_input,))
exported_model = exported_program.module()
with torch.inference_mode():
    eager_output = baseline_model(export_input)
    exported_output = exported_model(export_input)
assert torch.allclose(eager_output, exported_output, atol=1e-6, rtol=1e-5)

# The eager backend exercises Dynamo graph capture without claiming a native-kernel speedup.
compiled_model = torch.compile(copy.deepcopy(baseline_model), backend="eager", fullgraph=True)
with torch.inference_mode():
    compiled_output = compiled_model(export_input)
assert torch.allclose(eager_output, compiled_output, atol=1e-6, rtol=1e-5)
graph_ops = [node.target for node in exported_program.graph.nodes if node.op == "call_function"]
print({"captured_ops": len(graph_ops), "export_parity": True, "compile_parity": True})
```

</details>

A production test matrix should cover minimum, typical, and maximum shapes; empty or malformed inputs; numerically difficult values; and every declared device/precision profile. Track graph breaks, recompilation count, fallback operators, peak workspace, engine build time, and cold-start latency in addition to warm throughput.


### **ONNX, TensorRT, and Triton** {#onnx-tensorrt-triton}

These names occupy different layers:

- **ONNX** is an operator graph format and interchange contract. **ONNX Runtime** validates and executes that graph, partitions it among hardware-specific execution providers, and applies graph optimizations.
- **TensorRT** builds NVIDIA-specific optimized inference engines. Precision choices, workspace, tactic selection, and optimization profiles can make the artifact hardware- and shape-specific.
- **Triton Inference Server** is a serving layer. It manages model repositories, protocols, instances, metrics, ensembles, and schedulers; it can host TensorRT, ONNX Runtime, and other backends.

![The relationship among Triton, ONNX Runtime, TensorRT, and hardware kernels.](assets/dl20-runtime-stack.svg){fig-align="center" width="74%" fig-alt="Triton forms the request-serving layer above ONNX Runtime and TensorRT execution paths, which both target hardware kernels and memory."}

[ONNX Runtime](https://onnxruntime.ai/docs/) emphasizes cross-framework and cross-language execution. [TensorRT dynamic-shape documentation](https://docs.nvidia.com/deeplearning/tensorrt/latest/inference-library/work-with-dynamic-shapes.html) requires optimization profiles that bound each runtime dimension. The [Triton dynamic batcher](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/batcher.html) combines compatible stateless requests, while [Model Analyzer](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/model_analyzer/README.html) searches configuration trade-offs.

<details>
<summary><strong>Python: validate shape profiles and generate a minimal Triton serving contract</strong></summary>

```python
deployment_manifest = {
    "name": "digits_mlp",
    "version": 1,
    "input": {"name": "pixels", "dtype": "FP32", "shape": [64]},
    "output": {"name": "logits", "dtype": "FP32", "shape": [10]},
    "batch_profile": {"min": 1, "opt": 16, "max": 64},
}


def validate_batch_profile(manifest, observed_batches):
    profile = manifest["batch_profile"]
    if not (1 <= profile["min"] <= profile["opt"] <= profile["max"]):
        raise ValueError("invalid optimization profile ordering")
    unsupported = [batch for batch in observed_batches if not profile["min"] <= batch <= profile["max"]]
    return unsupported


observed = [1, 8, 16, 32, 64]
assert validate_batch_profile(deployment_manifest, observed) == []
triton_config = f'''name: "{deployment_manifest["name"]}"
platform: "onnxruntime_onnx"
max_batch_size: {deployment_manifest["batch_profile"]["max"]}
input [{{ name: "pixels" data_type: TYPE_FP32 dims: [64] }}]
output [{{ name: "logits" data_type: TYPE_FP32 dims: [10] }}]
dynamic_batching {{ preferred_batch_size: [8, 16, 32] max_queue_delay_microseconds: 800 }}'''
assert "dynamic_batching" in triton_config and "dims: [64]" in triton_config
print(triton_config)
```

</details>

An exported graph is not deployment completion. Compare eager and runtime outputs, verify that expected nodes were assigned to the accelerator rather than a CPU fallback, inspect engine profiles against live shapes, and load-test the server with realistic concurrency. Version the model, preprocessing, runtime, driver, and configuration together because any of them can change behavior or performance.


### **Cloud, Edge, and On-Device Deployment** {#cloud-edge-on-device}

Cloud accelerators offer elastic capacity, mature observability, and large memory, but every request pays network and tenancy costs. An edge gateway moves computation near the data source and can aggregate several devices under a fixed local budget. On-device execution minimizes network dependence and can improve privacy, yet faces strict memory, energy, thermal, binary-size, and operator constraints.

![Cloud, edge, and on-device constraints leading to artifact selection.](assets/dl20-deployment-targets.svg){fig-align="center" width="72%" fig-alt="Cloud, edge, and on-device deployment boxes list their distinct constraints and feed into a shared artifact selection rule."}

Placement is an end-to-end decision. A tiny model may save server compute but lose its benefit when radio startup dominates latency. A larger local model may exceed thermal limits during sustained use. Hybrid designs can run a local confidence gate and escalate uncertain requests, but that creates consistency, privacy, and fallback questions. Package size, peak working memory, cold start, energy per request, offline availability, update bandwidth, and target-specific operator coverage belong beside accuracy.

<details>
<summary><strong>Python: select candidate artifacts under explicit deployment budgets</strong></summary>

```python
def parameter_bytes(model):
    return sum(parameter.numel() * parameter.element_size() for parameter in model.parameters())


# Logical compressed sizes are separated from the FP32 teaching tensors used to emulate numerics.
ptq_logical_bytes = weight_count + 4 * (bias_count + 3 + baseline_model.fc1.out_features + baseline_model.fc2.out_features + baseline_model.fc3.out_features)
candidates = [
    {"name": "FP32_teacher", "accuracy": baseline_accuracy, "bytes": parameter_bytes(baseline_model)},
    {"name": "PTQ_W8A8", "accuracy": ptq_accuracy, "bytes": ptq_logical_bytes},
    {"name": "INT4_weight_only", "accuracy": int4_accuracy, "bytes": int4_logical_bytes},
    {"name": "distilled_student", "accuracy": student_accuracy, "bytes": parameter_bytes(student_model)},
]


def feasible_artifacts(items, minimum_accuracy, maximum_kib):
    return [item for item in items if item["accuracy"] >= minimum_accuracy and item["bytes"] / 1024 <= maximum_kib]


edge_choices = feasible_artifacts(candidates, minimum_accuracy=0.90, maximum_kib=40)
assert edge_choices and all(item["bytes"] <= 40 * 1024 for item in edge_choices)
print([{"name": item["name"], "accuracy": round(item["accuracy"], 3), "KiB": round(item["bytes"] / 1024, 1)} for item in edge_choices])
```

</details>

This first gate only filters accuracy and logical artifact size. A release decision still needs target-device latency, peak resident memory, energy, thermal behavior, startup time, and signed update/rollback tests. Never infer mobile speed from desktop CPU timing or logical bit width.


### **Monitoring, Drift, Rollback, and Cost Control** {#monitoring-drift-rollback-cost}

Deployment changes the evidence available to the model team. Inputs and system signals arrive immediately, while trustworthy labels may be delayed or absent. Monitoring therefore separates four layers: **service health** (errors, saturation, latency), **data health** (schema, missingness, range, drift), **model behavior** (confidence, abstention, slice metrics), and **business/safety outcomes** (the delayed consequence that motivated the model).

![A monitoring loop from traffic through telemetry and rollback decisions.](assets/dl20-monitoring-loop.svg){fig-align="center" width="72%" fig-alt="Traffic flows through a model into telemetry and delayed quality assessment, then a decision node can continue, canary, or roll back."}

Distribution drift is evidence of change, not proof of quality loss. Jensen-Shannon divergence between reference and live histograms is bounded and symmetric:

$$JS(P,Q)=\frac{1}{2}D_{KL}(P\|M)+\frac{1}{2}D_{KL}(Q\|M), \qquad M=\frac{P+Q}{2}.$$

A large (JS) says feature frequencies changed; only delayed labels, controlled audits, or task-specific proxies establish whether predictions degraded. Monitor slices because aggregate stability can hide a failing device, locale, class, or sequence-length band.

![Clean Digits requests and a synthetic brightness and noise shift.](assets/dl20-digits-clean-drift.png){fig-align="center" width="72%" fig-alt="Two rows compare six clean handwritten digit images with brightness-shifted and noisy versions used to demonstrate input drift."}

*Data source: Alpaydin and Kaynak, [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49), CC BY 4.0. The lower row applies a locally generated brightness/noise shift for diagnosis.*

<details>
<summary><strong>PyTorch: connect drift, confidence, quality, and a rollback policy</strong></summary>

```python
test_y = targets[test_ids]
drift_generator = torch.Generator().manual_seed(2020)
shifted_x = (0.55 * test_x + 0.25 + 0.12 * torch.randn(test_x.shape, generator=drift_generator)).clamp(0, 1)


def prediction_snapshot(model, x, y):
    model.eval()
    with torch.inference_mode():
        probabilities = model(x).softmax(dim=-1)
    return {
        "accuracy": float((probabilities.argmax(1) == y).float().mean()),
        "mean_confidence": float(probabilities.max(dim=1).values.mean()),
    }


def jensen_shannon_histogram(reference, live, bins=20):
    p, _ = np.histogram(reference.numpy(), bins=bins, range=(0, 1), density=False)
    q, _ = np.histogram(live.numpy(), bins=bins, range=(0, 1), density=False)
    p = (p + 1e-8) / (p.sum() + bins * 1e-8)
    q = (q + 1e-8) / (q.sum() + bins * 1e-8)
    m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


clean_snapshot = prediction_snapshot(baseline_model, test_x, test_y)
shifted_snapshot = prediction_snapshot(baseline_model, shifted_x, test_y)
drift_score = jensen_shannon_histogram(features[train_ids].flatten(), shifted_x.flatten())
rollback = shifted_snapshot["accuracy"] < clean_snapshot["accuracy"] - 0.08
assert drift_score > 0 and shifted_snapshot["accuracy"] <= clean_snapshot["accuracy"]
print({"clean": clean_snapshot, "shifted": shifted_snapshot, "JS": round(drift_score, 4), "rollback": rollback})
```

</details>

A safe rollout uses shadow traffic, offline replay, a small canary, and explicit automatic/manual rollback thresholds. Preserve the previous artifact and preprocessing bundle, verify rollback before launch, and bound spend with admission limits, output-length caps, cache quotas, autoscaling limits, and cost per successful request. Cost per raw token is incomplete if retries, failures, and poor-quality outputs rise.


### **The Accuracy-Latency-Memory Trade-Off** {#accuracy-latency-memory-tradeoff}

Deployment optimization is multi-objective. Candidate (A) **dominates** (B) if it is at least as accurate, no slower, and no larger, with a strict improvement in at least one dimension. Non-dominated candidates form a Pareto frontier; the product's constraints select a point on that frontier. A weighted score can hide an unacceptable threshold, so first enforce hard quality, safety, memory, and tail-latency requirements, then optimize cost among feasible artifacts.

![A conceptual Pareto frontier for deployment candidates.](assets/dl20-pareto.svg){fig-align="center" width="70%" fig-alt="Accuracy is plotted against latency and memory cost; several candidates lie on a dashed Pareto frontier while one candidate is dominated."}

Compression methods move different coordinates. PTQ and weight-only quantization target representation and bandwidth. QAT recovers quality under a chosen quantizer. Pruning needs structural or sparse-kernel support. Distillation changes the architecture. Batching improves utilization but adds queue delay. KV caching spends memory to avoid repeated computation. No method should be selected by a single file-size or microbenchmark number.

<details>
<summary><strong>Python: build a reproducible candidate table and identify non-dominated options</strong></summary>

```python
model_variants = {
    "FP32_teacher": baseline_model,
    "PTQ_W8A8": ptq_model,
    "QAT_fake_quant": qat_model,
    "INT4_simulation": int4_model,
    "pruned_dense_shape": pruned_model,
    "distilled_student": student_model,
}
logical_bytes = {
    "FP32_teacher": parameter_bytes(baseline_model),
    "PTQ_W8A8": ptq_logical_bytes,
    "QAT_fake_quant": ptq_logical_bytes,
    "INT4_simulation": int4_logical_bytes,
    # Unstructured zeros do not shrink a dense artifact without a sparse format.
    "pruned_dense_shape": parameter_bytes(pruned_model),
    "distilled_student": parameter_bytes(student_model),
}

deployment_records = []
for name, model in model_variants.items():
    timing = benchmark_classifier(model, test_x[:128], batch_size=16, repeats=2)
    deployment_records.append({
        "name": name,
        "accuracy": accuracy(model, test_loader),
        "latency_ms": timing["p50_batch_ms"],
        "logical_KiB": logical_bytes[name] / 1024,
    })


def dominates(a, b):
    no_worse = (
        a["accuracy"] >= b["accuracy"]
        and a["latency_ms"] <= b["latency_ms"]
        and a["logical_KiB"] <= b["logical_KiB"]
    )
    strictly_better = (
        a["accuracy"] > b["accuracy"]
        or a["latency_ms"] < b["latency_ms"]
        or a["logical_KiB"] < b["logical_KiB"]
    )
    return no_worse and strictly_better


frontier = [record for record in deployment_records if not any(dominates(other, record) for other in deployment_records if other is not record)]
assert frontier and all(record["accuracy"] > 0.80 for record in deployment_records)
print("all candidates:", [{**r, "accuracy": round(r["accuracy"], 3), "latency_ms": round(r["latency_ms"], 4), "logical_KiB": round(r["logical_KiB"], 1)} for r in deployment_records])
print("local reference frontier:", [record["name"] for record in frontier])
```

</details>

The low-bit variants above execute reconstructed FP32 or fake-quantized operations, so the local timing column is a **reference-path diagnostic**, not an INT8/INT4 kernel benchmark. Rebuild the table on each target runtime with real packed artifacts, realistic concurrency, p95/p99 latency, peak memory, energy, and slice quality. Hardware and runtime upgrades can change the frontier without any model-weight change.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Efficient deployment begins with a trustworthy eager reference and ends with a monitored, reversible release. The middle is a chain of transformations, each of which changes assumptions and therefore requires evidence.

| Decision | Primary resource | Main benefit | Common failure | Required evidence |
|---|---|---|---|---|
| Dynamic/continuous batching | scheduler slots | higher utilization | queueing and starvation | latency distribution under live-like arrivals |
| KV cache / paging | accelerator memory | avoids repeated prefix projections | fragmentation, eviction, wrong positions | token-level parity and cache telemetry |
| Speculative decoding | target model calls | more accepted tokens per target step | weak draft or incorrect correction | distributional tests and TTFT/TPOT |
| PTQ / QAT | value representation | smaller artifacts and supported low-bit compute | clipping and sensitive layers | quality, layer error, packed-kernel benchmark |
| Weight-only quantization | weight bandwidth | lower decode memory traffic | fallback/dequantization overhead | target tokens/s and resident memory |
| Pruning | parameter structure | potential sparse or smaller graph | zeros without kernel speedup | actual sparse/structured runtime measurement |
| Distillation | architecture size | smaller dense student | copied errors or lost tail behavior | hard-label baseline and slice evaluation |
| Export / compilation | graph and kernels | fusion, portability, AOT execution | graph break, unsupported op, shape miss | parity matrix and fallback report |
| Monitoring / rollback | operational risk | detects and limits regressions | proxy drift mistaken for quality | telemetry, delayed labels, tested rollback |

<details>
<summary><strong>Python: encode a minimum release gate</strong></summary>

```python
def release_gate(candidate, reference_accuracy, maximum_accuracy_drop=0.02, maximum_kib=80):
    checks = {
        "quality": candidate["accuracy"] >= reference_accuracy - maximum_accuracy_drop,
        "memory": candidate["logical_KiB"] <= maximum_kib,
        "latency_measured": candidate["latency_ms"] > 0,
        "rollback_artifact_present": True,
    }
    return checks, all(checks.values())


candidate = next(record for record in deployment_records if record["name"] == "distilled_student")
checks, releasable_in_local_gate = release_gate(candidate, baseline_accuracy, maximum_accuracy_drop=0.04)
assert set(checks) == {"quality", "memory", "latency_measured", "rollback_artifact_present"}
print({"candidate": candidate["name"], "checks": checks, "local_gate": releasable_in_local_gate})
```

</details>

A practical sequence is: define the request and quality contract; establish warm and tail baselines; profile the bottleneck; apply one intervention; validate numerical and task parity; benchmark the target runtime; canary with observability; and preserve rollback. Optimize the constrained system, not an isolated kernel, bit width, or average.
